In [1]:
import ast
import math
import pandas as pd
import osmnx as ox
from shapely.geometry import Point, LineString, MultiLineString
from shapely.ops import unary_union

DATA_FILE = "osm_segment_metadata.csv"
GRAPH_FILE = "bhubaneswar_drive.graphml"

print("Loading dataset...")
df = pd.read_csv(DATA_FILE)

print("Loading OSM graph...")
G = ox.load_graphml(GRAPH_FILE)


# =========================================================
# 1. Parse OSM IDs
# =========================================================

def parse_osmid(value):

    if pd.isna(value):
        return []

    value = str(value).strip()

    if value.startswith("[") and value.endswith("]"):
        try:
            return [str(x) for x in ast.literal_eval(value)]
        except Exception:
            return []

    return [value]


df["osmid_list"] = df["osmid"].apply(parse_osmid)


# =========================================================
# 2. Build OSM ID -> ALL graph edges lookup
# =========================================================

osm_edges = {}

for u, v, k, data in G.edges(keys=True, data=True):

    osmid = data.get("osmid")

    if osmid is None:
        continue

    if isinstance(osmid, list):
        ids = [str(x) for x in osmid]
    else:
        ids = [str(osmid)]

    for oid in ids:

        if oid not in osm_edges:
            osm_edges[oid] = []

        osm_edges[oid].append({
            "u": u,
            "v": v,
            "key": k,
            "length": float(data.get("length", 0)),
            "highway": data.get("highway"),
            "geometry": data.get("geometry")
        })


# =========================================================
# 3. Haversine
# =========================================================

def haversine(lat1, lon1, lat2, lon2):

    R = 6371000

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.asin(math.sqrt(a))


# =========================================================
# 4. Validation
# =========================================================

results = []

for _, row in df.iterrows():

    segment_id = row["segment_id"]

    dataset_ids = row["osmid_list"]

    dataset_length = float(row["length_m"])
    dataset_lat = float(row["lat"])
    dataset_lon = float(row["lon"])
    dataset_type = str(row["road_type"])

    # -----------------------------------------------------
    # Collect ALL matching graph edges
    # -----------------------------------------------------

    candidate_edges = []

    for oid in dataset_ids:

        if oid in osm_edges:
            candidate_edges.extend(osm_edges[oid])

    # Remove duplicate graph edges
    unique_edges = {}

    for edge in candidate_edges:

        key = (
            edge["u"],
            edge["v"],
            edge["key"]
        )

        unique_edges[key] = edge

    candidate_edges = list(unique_edges.values())

    if not candidate_edges:

        results.append({
            "segment_id": segment_id,
            "matched": False
        })

        continue

    # -----------------------------------------------------
    # Aggregate OSM length
    # -----------------------------------------------------

    osm_total_length = sum(
        edge["length"]
        for edge in candidate_edges
    )

    length_error = abs(
        dataset_length - osm_total_length
    )

    length_error_pct = (
        length_error / dataset_length * 100
        if dataset_length > 0
        else None
    )

    # -----------------------------------------------------
    # Geographic consistency
    #
    # Compare dataset coordinate to the centroid of
    # matched OSM edge geometries / nodes.
    # -----------------------------------------------------

    geometries = []

    for edge in candidate_edges:

        geometry = edge["geometry"]

        if geometry is not None:
            geometries.append(geometry)

        else:
            # Construct edge geometry from nodes
            u_data = G.nodes[edge["u"]]
            v_data = G.nodes[edge["v"]]

            geometries.append(
                LineString([
                    (
                        float(u_data["x"]),
                        float(u_data["y"])
                    ),
                    (
                        float(v_data["x"]),
                        float(v_data["y"])
                    )
                ])
            )

    try:

        combined_geometry = unary_union(geometries)

        centroid = combined_geometry.centroid

        osm_lon = centroid.x
        osm_lat = centroid.y

        coordinate_distance = haversine(
            dataset_lat,
            dataset_lon,
            osm_lat,
            osm_lon
        )

    except Exception:

        coordinate_distance = None
        osm_lat = None
        osm_lon = None

    # -----------------------------------------------------
    # Road types
    # -----------------------------------------------------

    highway_types = []

    for edge in candidate_edges:

        highway = edge["highway"]

        if isinstance(highway, list):
            highway_types.extend(
                [str(x) for x in highway]
            )
        elif highway is not None:
            highway_types.append(str(highway))

    highway_types = sorted(set(highway_types))

    results.append({

        "segment_id": segment_id,

        "matched": True,

        "num_dataset_osm_ids": len(dataset_ids),

        "num_graph_edges": len(candidate_edges),

        "dataset_length_m": dataset_length,

        "osm_total_length_m": osm_total_length,

        "length_error_m": length_error,

        "length_error_pct": length_error_pct,

        "dataset_lat": dataset_lat,

        "dataset_lon": dataset_lon,

        "osm_centroid_lat": osm_lat,

        "osm_centroid_lon": osm_lon,

        "coordinate_distance_m": coordinate_distance,

        "dataset_road_type": dataset_type,

        "osm_road_types": ", ".join(highway_types),

        "matched_osm_ids": ", ".join(dataset_ids)

    })


# =========================================================
# 5. Results
# =========================================================

results_df = pd.DataFrame(results)

matched = results_df[
    results_df["matched"] == True
].copy()

print("\n========================================")
print("AGGREGATED OSM VALIDATION")
print("========================================")

print("\nTotal segments:", len(results_df))
print("Matched:", len(matched))

print(
    "Match rate:",
    f"{len(matched) / len(results_df) * 100:.2f}%"
)


# =========================================================
# LENGTH
# =========================================================

print("\n========== AGGREGATED LENGTH ==========")

print(
    "Mean error:",
    f"{matched['length_error_m'].mean():.2f} m"
)

print(
    "Median error:",
    f"{matched['length_error_m'].median():.2f} m"
)

print(
    "Mean percentage error:",
    f"{matched['length_error_pct'].mean():.2f}%"
)

print(
    "Median percentage error:",
    f"{matched['length_error_pct'].median():.2f}%"
)

print(
    "Within 5%:",
    f"{(matched['length_error_pct'] <= 5).sum()} / {len(matched)}"
)

print(
    "Within 10%:",
    f"{(matched['length_error_pct'] <= 10).sum()} / {len(matched)}"
)

print(
    "Within 20%:",
    f"{(matched['length_error_pct'] <= 20).sum()} / {len(matched)}"
)


# =========================================================
# COORDINATES
# =========================================================

print("\n========== AGGREGATED COORDINATES ==========")

print(
    "Mean distance:",
    f"{matched['coordinate_distance_m'].mean():.2f} m"
)

print(
    "Median distance:",
    f"{matched['coordinate_distance_m'].median():.2f} m"
)

print(
    "Within 50 m:",
    f"{(matched['coordinate_distance_m'] <= 50).sum()} / {len(matched)}"
)

print(
    "Within 100 m:",
    f"{(matched['coordinate_distance_m'] <= 100).sum()} / {len(matched)}"
)

print(
    "Within 500 m:",
    f"{(matched['coordinate_distance_m'] <= 500).sum()} / {len(matched)}"
)


# =========================================================
# MULTI-OSM SEGMENTS
# =========================================================

print("\n========== OSM COMPOSITION ==========")

print(
    "Segments with multiple OSM IDs:",
    (matched["num_dataset_osm_ids"] > 1).sum()
)

print(
    "Segments with single OSM ID:",
    (matched["num_dataset_osm_ids"] == 1).sum()
)


# =========================================================
# WORST LENGTH MISMATCHES
# =========================================================

print("\n========== WORST LENGTH MISMATCHES ==========")

worst = matched.sort_values(
    "length_error_pct",
    ascending=False
).head(15)

print(
    worst[
        [
            "segment_id",
            "num_dataset_osm_ids",
            "dataset_length_m",
            "osm_total_length_m",
            "length_error_m",
            "length_error_pct",
            "matched_osm_ids"
        ]
    ].to_string(index=False)
)


# =========================================================
# SAVE
# =========================================================

output_file = "osm_aggregated_validation.csv"

results_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:", output_file)

Loading dataset...
Loading OSM graph...

AGGREGATED OSM VALIDATION

Total segments: 700
Matched: 700
Match rate: 100.00%

========== AGGREGATED LENGTH ==========
Mean error: 974.39 m
Median error: 771.79 m
Mean percentage error: 749.41%
Median percentage error: 515.09%
Within 5%: 30 / 700
Within 10%: 33 / 700
Within 20%: 44 / 700

========== AGGREGATED COORDINATES ==========
Mean distance: 244.92 m
Median distance: 170.59 m
Within 50 m: 142 / 700
Within 100 m: 245 / 700
Within 500 m: 610 / 700

========== OSM COMPOSITION ==========
Segments with multiple OSM IDs: 87
Segments with single OSM ID: 613

========== WORST LENGTH MISMATCHES ==========
 segment_id  num_dataset_osm_ids  dataset_length_m  osm_total_length_m  length_error_m  length_error_pct                  matched_osm_ids
        699                    1         60.368973         3744.521826     3684.152853       6102.725758                        865024248
        639                    1         68.470695         3744.521826 